In [3]:
import os
import pydicom
import numpy as np
import matplotlib.pyplot as plt

print("import libraries successfully")

import libraries successfully


In [6]:
def extraction_ct_patient(base_folder):
    ct_patient_dict = {}
    extracted_files = 0
    
    print(f"Start search in folder: {base_folder}...\n")
    
    for root, dirs, files in os.walk(base_folder):
        for file in files:
            if file.lower().endswith('.dcm'):
                percorso_completo = os.path.join(root, file)
                extracted_files += 1
                
                try:
                    ds = pydicom.dcmread(percorso_completo, stop_before_pixels=True)
                    
                    modality = ds.get('Modality', '').upper()
                    body_part = ds.get('BodyPartExamined', '').upper()
                    series_desc = ds.get('SeriesDescription', '').upper()
                    patient_id = str(ds.get('PatientID', 'ID_Sconosciuto'))
                    
                    # Filter by modality and body part
                    is_ct = (modality == 'CT')
                    chest_keyword = ['CHEST', 'THORAX', 'LUNG', 'TORACE']
                    is_chest = any(k in body_part or k in series_desc for k in chest_keyword)
                    
                    # Add to dictionary if it's a chest CT
                    if is_ct and is_chest:
                        if patient_id not in ct_patient_dict:
                            ct_patient_dict[patient_id] = []
                            print(f"Find new patient: {patient_id}")
                        
                        ct_patient_dict[patient_id].append(percorso_completo)
                        
                except Exception:
                    pass # Ignora file corrotti
                    
    print(f"\nFinish! Analised {extracted_files} file .dcm total.")
    return ct_patient_dict